# ViNewsQA — Qwen2.5-1.5B: train 4 adapters (Unsloth)

Train tuần tự trên dữ liệu SQuAD 1.1 tiếng Việt, cùng prompt/seed/cấu hình:

1. **LoRA** → `qwen2.5-1.5b-instruct-lora-vinewsqa`
2. **TinyLoRA** → `qwen2.5-1.5b-instruct-tinylora-vinewsqa`
3. **DoRA** → `qwen2.5-1.5b-instruct-dora-vinewsqa`
4. **DeLoRA** → `qwen2.5-1.5b-instruct-delora-vinewsqa`

Train dùng đáp án tham chiếu đầu tiên. Early stopping theo **dev loss**; đánh giá cuối trên toàn bộ **test** bằng EM/F1 lấy giá trị lớn nhất trên mọi đáp án tham chiếu.

### Cách chạy
1. Chạy các pip cells rồi **Restart Kernel**.
2. Chạy tuần tự config, data, profiling, format và train loop.
3. Cell eval tạo `eval_infer_subprocess.py`, chạy Transformers + PEFT thuần (không Unsloth).
4. Cell cuối lưu `eval_compare_adapters_vinewsqa_test.json`; tùy chọn xuất `results.json`.

In [ ]:
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

In [ ]:
# Cài dependencies trước, sau đó pin PEFT đúng bản có cả TinyLoRA và DeLoRA.
!pip install transformers trl accelerate bitsandbytes xformers datasets --no-cache-dir
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-cache-dir
!pip install unsloth_zoo scikit-learn safetensors --no-cache-dir
!pip install "peft==0.19.1" --no-deps --force-reinstall --no-cache-dir

In [ ]:
import importlib
import inspect

for pkg in ["torch", "transformers", "datasets", "unsloth", "peft"]:
    importlib.import_module(pkg)
    print(f"OK  {pkg}")

import peft
from peft import DeloraConfig, TinyLoraConfig

print(f"PEFT version: {peft.__version__}")
if peft.__version__ != "0.19.1":
    raise RuntimeError(f"Cần peft==0.19.1, hiện tại là {peft.__version__}. Restart Kernel.")
for cls, required in [(DeloraConfig, {"r", "delora_lambda", "target_modules", "module_dropout"}), (TinyLoraConfig, {"r", "target_modules"})]:
    sig = inspect.signature(cls.__init__).parameters
    missing = required.difference(sig)
    if missing:
        raise RuntimeError(f"{cls.__name__} thiếu tham số: {sorted(missing)}")
    print(f"{cls.__name__} API OK:", ", ".join(k for k in sig if k != "self"))

In [ ]:
import warnings, logging, os
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["ACCELERATE_BYPASS_DEVICE_MAP"] = "true"
os.environ["ACCELERATE_NUM_PROCESSES"] = "1"
os.environ["WORLD_SIZE"] = "1"
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"
os.environ["MASTER_ADDR"] = "127.0.0.1"
os.environ["MASTER_PORT"] = "29500"
for _n in ("transformers", "datasets", "torch", "unsloth", "peft", "accelerate", "huggingface_hub"):
    logging.getLogger(_n).setLevel(logging.ERROR)
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass
print("Env OK | PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True | V5")

In [ ]:
import json, math, gc, re, string, unicodedata, subprocess
from collections import Counter
from pathlib import Path
import torch
from datasets import Dataset
from tqdm import tqdm
from transformers import AutoTokenizer

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
NOTEBOOK_VERSION = "V5.1-ViNewsQA"
print(f"NOTEBOOK_VERSION = {NOTEBOOK_VERSION}  (train cell phải in >>> TRAIN_FIX_V5 <<<)")
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM_PROMPT = ("Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. Chỉ trả về đúng cụm từ xuất hiện trong đoạn văn, không giải thích, không viết câu hoàn chỉnh và không thêm tiền tố như 'Đáp án:'.\n\nĐoạn văn:\n{context}")
DATASET_ROOT = Path("../../")
TRAIN_JSON_PATH = DATASET_ROOT / "train_ViNewsQA.json"
DEV_JSON_PATH = DATASET_ROOT / "dev_ViNewsQA.json"
TEST_JSON_PATH = DATASET_ROOT / "test_ViNewsQA.json"
PROFILING_CONFIG_PATH = "profiling_config.json"
COMPARE_EVAL_PATH = "eval_compare_adapters_vinewsqa_test.json"
RUN_TRAINING = True
RUN_METRIC_EVAL = True
RUN_SUBMISSION_EXPORT = False
LOAD_IN_4BIT, LOAD_IN_8BIT = True, False

def load_in_4bit_for_method(method_name):
    """DeLoRA tính norm trên base.weight nên không tương thích trọng số BnB 4-bit."""
    return False if method_name == "delora" else LOAD_IN_4BIT

MAX_SEQ_CAP, MIN_SEQ_LENGTH, MAX_NEW_TOKENS = 4096, 512, 64
MIN_FREE_VRAM_GIB, INFER_BATCH_SIZE, EXPECTED_TEST_SIZE = 8.0, 16, 1987
EVAL_SPLIT, EVAL_MAX_SAMPLES, EVAL_LOG_EVERY = "test", None, 50
SMOKE_TEST, SMOKE_TRAIN_SAMPLES, SMOKE_EVAL_SAMPLES = False, 200, 50
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
USE_EARLY_STOPPING = True
MAX_TRAIN_EPOCHS, EARLY_STOPPING_PATIENCE = 5, 3
EARLY_STOPPING_THRESHOLD, EVAL_STEPS, SAVE_STEPS, SAVE_TOTAL_LIMIT = 0.001, 200, 200, 3
TRAIN_COMMON = dict(per_device_train_batch_size=2, gradient_accumulation_steps=4, warmup_steps=10, num_train_epochs=MAX_TRAIN_EPOCHS, learning_rate=2e-4, optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="cosine", seed=3407)
ADAPTER_VARIANTS = [
    {"name":"lora", "save_path":"qwen2.5-1.5b-instruct-lora-vinewsqa", "output_dir":"outputs_vinewsqa_lora"},
    {"name":"tinylora", "save_path":"qwen2.5-1.5b-instruct-tinylora-vinewsqa", "output_dir":"outputs_vinewsqa_tinylora"},
    {"name":"dora", "save_path":"qwen2.5-1.5b-instruct-dora-vinewsqa", "output_dir":"outputs_vinewsqa_dora"},
    {"name":"delora", "save_path":"qwen2.5-1.5b-instruct-delora-vinewsqa", "output_dir":"outputs_vinewsqa_delora"},
]
TRAIN_METHODS = EVAL_METHODS = ["lora", "tinylora", "dora", "delora"]
TQDM_BAR = "{desc}: {percentage:3.0f}%|{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"

def build_messages(context, question, answer=None, for_inference=False):
    messages = [{"role":"system", "content":SYSTEM_PROMPT.format(context=context)}, {"role":"user", "content":question}]
    if answer is not None and not for_inference: messages.append({"role":"assistant", "content":answer})
    return messages

def sample_to_train_text(sample, tok):
    return tok.apply_chat_template(build_messages(sample["context"], sample["question"], sample["answer"]), tokenize=False, add_generation_prompt=False)

def load_tokenizer(model_path=BASE_MODEL_NAME):
    tok = AutoTokenizer.from_pretrained(model_path)
    if tok.chat_template is None: raise RuntimeError(f"Tokenizer {model_path} thiếu chat_template.")
    return tok

def clear_gpu(verbose=False):
    gc.collect()
    if not torch.cuda.is_available(): return
    try: torch.cuda.synchronize()
    except Exception: pass
    torch.cuda.empty_cache()
    try: torch.cuda.ipc_collect()
    except Exception: pass
    gc.collect(); torch.cuda.empty_cache()
    if verbose: report_vram("after clear_gpu")

def report_vram(label="VRAM"):
    if not torch.cuda.is_available():
        print(f"[{label}] CUDA not available"); return 0.0
    free, total = torch.cuda.mem_get_info(); free_gib, total_gib = free/1024**3, total/1024**3
    print(f"[{label}] driver_used={total_gib-free_gib:.2f} GiB | free={free_gib:.2f}/{total_gib:.2f} GiB | pytorch_alloc={torch.cuda.memory_allocated()/1024**3:.3f} GiB", flush=True)
    return free_gib

def release_stale_training_objects():
    g = globals()
    for name in ("model", "tokenizer", "trainer", "model_eval", "tokenizer_eval", "tokenizer_prof", "trained_paths"):
        if name in g:
            try: del g[name]
            except Exception: pass
    clear_gpu()

def preflight_vram(min_free_gib=None):
    min_free = MIN_FREE_VRAM_GIB if min_free_gib is None else min_free_gib
    release_stale_training_objects(); clear_gpu(verbose=True); free_gib = report_vram("preflight")
    try:
        out = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"], text=True)
        print(f"[preflight] nvidia-smi used/total MiB: {out.strip()}", flush=True)
    except Exception as e: print(f"[preflight] nvidia-smi skip: {e}", flush=True)
    if free_gib < min_free: raise RuntimeError(f"GPU không đủ VRAM: free={free_gib:.2f} GiB, cần >= {min_free} GiB.")
    print(f"✅ preflight OK | free={free_gib:.2f} GiB", flush=True); return free_gib

In [ ]:
def load_vinewsqa_split(path):
    """Load SQuAD 1.1 JSON; SFT uses first answer, metrics retain every reference."""
    with open(path, encoding="utf-8") as f: payload = json.load(f)
    samples = []
    for article in payload["data"]:
        title = article.get("title", "")
        for paragraph in article["paragraphs"]:
            context = str(paragraph["context"]).strip()
            for qa in paragraph["qas"]:
                question = str(qa["question"]).strip()
                gold_answers = [str(a.get("text") or "").strip() for a in qa.get("answers", []) if str(a.get("text") or "").strip()]
                if not context or not question or not gold_answers: continue
                samples.append({"id":str(qa["id"]), "title":title, "context":context, "question":question, "answer":gold_answers[0], "gold_answers":gold_answers})
    return samples

train_samples = load_vinewsqa_split(TRAIN_JSON_PATH)
dev_samples = load_vinewsqa_split(DEV_JSON_PATH)
test_samples = load_vinewsqa_split(TEST_JSON_PATH)
expected = {"train":17563, "dev":2497, "test":EXPECTED_TEST_SIZE}
splits = {"train":train_samples, "dev":dev_samples, "test":test_samples}
for split, samples in splits.items():
    if len(samples) != expected[split]: raise RuntimeError(f"{split} phải có {expected[split]} QAs, nhận {len(samples)}")
    ids = [sample["id"] for sample in samples]
    if len(ids) != len(set(ids)): raise RuntimeError(f"{split} có id trùng")
    print(f"{split}: {len(samples)} samples")
if SMOKE_TEST:
    train_samples = train_samples[:SMOKE_TRAIN_SAMPLES]
    print(f"SMOKE_TEST train: {len(train_samples)} samples")
if EVAL_SPLIT == "test": eval_samples = test_samples
elif EVAL_SPLIT == "dev": eval_samples = dev_samples
else: raise ValueError(f"EVAL_SPLIT không hợp lệ: {EVAL_SPLIT}")
if SMOKE_TEST and EVAL_MAX_SAMPLES is None: EVAL_MAX_SAMPLES = SMOKE_EVAL_SAMPLES
print(f"Eval ({EVAL_SPLIT}): {len(eval_samples)} samples")

In [ ]:
if "train_samples" not in globals(): raise NameError("Chạy cell tải dataset trước.")

def compute_max_seq_length(samples, tok, cap=MAX_SEQ_CAP, min_len=MIN_SEQ_LENGTH):
    lengths = []
    pbar = tqdm(samples, total=len(samples), desc="Token profiling", unit="sample", bar_format=TQDM_BAR)
    for i, sample in enumerate(pbar, 1):
        lengths.append(len(tok.encode(sample_to_train_text(sample, tok))))
        pbar.set_postfix(done=f"{i}/{len(samples)}")
    lengths.sort(); n = len(lengths)
    stats = {"min":lengths[0], "p50":lengths[n//2], "p95":lengths[int(n*0.95)], "p99":lengths[int(n*0.99)], "max":lengths[-1]}
    chosen = max(((min(math.ceil(stats["p99"]*1.05), cap)+255)//256)*256, min_len)
    truncated = sum(length > chosen for length in lengths)
    stats.update({"chosen_max_seq_length":chosen, "truncated_samples":truncated, "truncated_pct":round(100*truncated/n, 3)})
    return chosen, stats

tokenizer_prof = load_tokenizer()
if Path(PROFILING_CONFIG_PATH).exists() and not RUN_TRAINING:
    with open(PROFILING_CONFIG_PATH, encoding="utf-8") as f: prof_cfg = json.load(f)
    max_seq_length, length_stats = prof_cfg["max_seq_length"], prof_cfg["token_length_stats"]
else:
    max_seq_length, length_stats = compute_max_seq_length(train_samples, tokenizer_prof)
    with open(PROFILING_CONFIG_PATH, "w", encoding="utf-8") as f: json.dump({"max_seq_length":max_seq_length, "token_length_stats":length_stats}, f, indent=2)
print(f"max_seq_length = {max_seq_length}")
for key, value in length_stats.items(): print(f"  {key}: {value}")
del tokenizer_prof
clear_gpu()

In [ ]:
tokenizer_fmt = load_tokenizer()

def formatting_prompts_func(examples):
    texts = []
    for context, question, answer in zip(examples["context"], examples["question"], examples["answer"]):
        texts.append(tokenizer_fmt.apply_chat_template(build_messages(context, question, answer), tokenize=False, add_generation_prompt=False))
    return {"text":texts}

train_hf = Dataset.from_list(train_samples)
dataset = train_hf.map(formatting_prompts_func, batched=True, remove_columns=train_hf.column_names)
print(f"Shared train dataset: {len(dataset)} samples")
eval_dataset = None
if USE_EARLY_STOPPING:
    dev_hf = Dataset.from_list(dev_samples)
    eval_dataset = dev_hf.map(formatting_prompts_func, batched=True, remove_columns=dev_hf.column_names)
    print(f"Dev dataset for early stopping: {len(eval_dataset)} samples")
print(dataset[0]["text"][:500])

## Train

Train tuần tự LoRA, TinyLoRA, DoRA và DeLoRA. Mọi variant dùng cùng train/dev split; `eval_loss` trên dev được kiểm tra mỗi 200 steps và early stopping có patience 3.

Cell train giữ cơ chế `TRAIN_FIX_V5`, preflight VRAM, dọn GPU và lưu adapter + tokenizer.

In [ ]:
def apply_adapter(model, method_name):
    """Attach LoRA, TinyLoRA, DoRA, or DeLoRA to the Unsloth-loaded base."""
    from unsloth import FastLanguageModel
    def _resolve_causallm(candidate):
        if hasattr(candidate, "prepare_inputs_for_generation"): return candidate
        if hasattr(candidate, "get_base_model"):
            base = candidate.get_base_model()
            if hasattr(base, "prepare_inputs_for_generation"): return base
        raise RuntimeError("Không tìm thấy ForCausalLM trên model Unsloth.")
    if method_name in ("lora", "dora"):
        model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, target_modules=TARGET_MODULES, lora_dropout=0.05, bias="none", use_gradient_checkpointing="unsloth", random_state=3407, use_dora=(method_name == "dora"))
        print(f"Applied: {method_name.upper()} (r=16, alpha=32)"); return model
    if method_name == "tinylora":
        import inspect, peft
        from peft import TinyLoraConfig, get_peft_model
        sig = inspect.signature(TinyLoraConfig.__init__).parameters
        desired = {"r":2, "u":64, "num_projections":64, "target_modules":TARGET_MODULES, "tinylora_dropout":0.0, "lora_dropout":0.0, "bias":"none", "task_type":"CAUSAL_LM", "weight_tying":0.0, "projection_seed":3407, "init_weights":True}
        config = TinyLoraConfig(**{k:v for k,v in desired.items() if k in sig})
        model = get_peft_model(_resolve_causallm(model), config)
        print(f"Applied: TinyLoRA (PEFT {peft.__version__})")
    elif method_name == "delora":
        import peft
        from peft import DeloraConfig, get_peft_model
        config = DeloraConfig(r=16, delora_lambda=15, target_modules=TARGET_MODULES, module_dropout=0.05, bias="none", task_type="CAUSAL_LM", init_weights=True)
        model = get_peft_model(_resolve_causallm(model), config)
        print(f"Applied: DeLoRA (PEFT {peft.__version__})")
    else:
        raise ValueError(f"Unknown method: {method_name}")
    model.config.use_cache = False
    if hasattr(model, "gradient_checkpointing_enable"): model.gradient_checkpointing_enable()
    model.print_trainable_parameters()
    return model

def _force_single_gpu_train_env():
    os.environ.update({"ACCELERATE_BYPASS_DEVICE_MAP":"true", "ACCELERATE_NUM_PROCESSES":"1", "WORLD_SIZE":"1", "LOCAL_WORLD_SIZE":"1", "LOCAL_RANK":"0", "RANK":"0"})

def _strip_hf_device_map(obj, seen=None):
    seen = set() if seen is None else seen
    if obj is None or id(obj) in seen: return
    seen.add(id(obj))
    if hasattr(obj, "hf_device_map"):
        try: delattr(obj, "hf_device_map")
        except Exception: obj.hf_device_map = None
    for attr in ("model", "base_model", "module", "pretrained_model"):
        child = getattr(obj, attr, None)
        if isinstance(child, torch.nn.Module): _strip_hf_device_map(child, seen)

def train_one_variant(variant, max_seq_length, dataset, eval_dataset=None):
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from trl import SFTTrainer
    from transformers import TrainingArguments, EarlyStoppingCallback
    import inspect
    print(">>> TRAIN_FIX_V5 <<<", flush=True)
    _force_single_gpu_train_env(); preflight_vram()
    name = variant["name"]; use_4bit = load_in_4bit_for_method(name)
    print(f"TRAIN VARIANT: {name.upper()} | 4bit={use_4bit}")
    model = tokenizer = trainer = None
    try:
        load_kwargs = dict(model_name=BASE_MODEL_NAME, max_seq_length=max_seq_length, dtype=None, load_in_4bit=use_4bit, load_in_8bit=LOAD_IN_8BIT)
        if "device_map" in inspect.signature(FastLanguageModel.from_pretrained).parameters: load_kwargs["device_map"] = None
        model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
        if torch.cuda.is_available(): model = model.to("cuda:0")
        model = apply_adapter(model, name); _strip_hf_device_map(model)
        args = dict(**TRAIN_COMMON, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(), output_dir=variant["output_dir"], logging_strategy="steps", logging_steps=SAVE_STEPS, save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=SAVE_TOTAL_LIMIT, report_to="none", dataloader_num_workers=0)
        params = inspect.signature(TrainingArguments.__init__).parameters
        eval_key = "eval_strategy" if "eval_strategy" in params else "evaluation_strategy"
        callbacks = []
        if USE_EARLY_STOPPING and eval_dataset is not None:
            args.update({eval_key:"steps", "eval_steps":EVAL_STEPS, "load_best_model_at_end":True, "metric_for_best_model":"eval_loss", "greater_is_better":False})
            callbacks = [EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE, early_stopping_threshold=EARLY_STOPPING_THRESHOLD)]
        else: args[eval_key] = "no"
        args = {k:v for k,v in args.items() if k in params or k == "output_dir"}
        sft = dict(model=model, train_dataset=dataset, eval_dataset=eval_dataset, args=TrainingArguments(**args), callbacks=callbacks)
        sft_params = inspect.signature(SFTTrainer.__init__).parameters
        sft["processing_class" if "processing_class" in sft_params else "tokenizer"] = tokenizer
        for key, value in (("dataset_text_field","text"), ("max_seq_length",max_seq_length), ("packing",False), ("dataset_num_proc",1)):
            if key in sft_params: sft[key] = value
        trainer = SFTTrainer(**sft); trainer.train()
        save_path = Path(variant["save_path"]); save_path.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(save_path, safe_serialization=True); tokenizer.save_pretrained(save_path)
        print(f"Saved adapter -> {save_path}"); return str(save_path)
    finally:
        if trainer is not None: del trainer
        if model is not None: del model
        if tokenizer is not None: del tokenizer
        clear_gpu(verbose=True)

In [ ]:
# === GPU DIAGNOSTIC (chạy trước train loop) ===
import subprocess as _sp
print("NOTEBOOK_VERSION =", NOTEBOOK_VERSION)
try: print(_sp.check_output(["nvidia-smi"], text=True))
except Exception as e: print("nvidia-smi:", e)
report_vram("diagnostic")
if torch.cuda.is_available():
    free_gib = torch.cuda.mem_get_info()[0] / 1024**3
    if free_gib < MIN_FREE_VRAM_GIB: raise RuntimeError(f"GPU zombie: free={free_gib:.2f} GiB. Restart kernel/container.")

In [ ]:
# === TRAIN LOOP ===
if RUN_TRAINING:
    preflight_vram()
    variant_map = {variant["name"]:variant for variant in ADAPTER_VARIANTS}
    trained_paths = {}
    for method in TRAIN_METHODS:
        if method not in variant_map: raise ValueError(f"Unknown method: {method}")
        print(f"\n>>> Starting {method} ...", flush=True)
        trained_paths[method] = train_one_variant(variant_map[method], max_seq_length, dataset, eval_dataset=eval_dataset if USE_EARLY_STOPPING else None)
        preflight_vram()
    print("\nTrain complete:")
    for method, path in trained_paths.items(): print(f"  {method}: {path}")
else:
    print("RUN_TRAINING=False — bỏ qua train.")

## ViNewsQA Evaluation — LoRA vs TinyLoRA vs DoRA vs DeLoRA

Đánh giá extractive QA trên test bằng subprocess sạch: Transformers + PEFT, không import Unsloth, batch inference với left padding. EM/F1 của mỗi mẫu là giá trị lớn nhất trên mọi gold reference.

In [ ]:
PREFIX_RE = re.compile(r"^(đáp án|answer|câu trả lời|theo đoạn văn|trong đoạn văn)\s*[:\-]?\s*", re.IGNORECASE)
ADAPTER_SUBMISSION_DIRS = {method:f"qwen2.5-1.5b-instruct-{method}-vinewsqa" for method in EVAL_METHODS}

def normalize_text(text):
    text = unicodedata.normalize("NFC", text or "")
    return " ".join(text.lower().translate(str.maketrans("", "", string.punctuation)).split())
def compute_em(pred, truth): return int(normalize_text(pred) == normalize_text(truth))
def compute_f1(pred, truth):
    pt, tt = normalize_text(pred).split(), normalize_text(truth).split()
    if not pt and not tt: return 1.0
    if not pt or not tt: return 0.0
    overlap = sum((Counter(pt) & Counter(tt)).values())
    if not overlap: return 0.0
    precision, recall = overlap/len(pt), overlap/len(tt)
    return 2*precision*recall/(precision+recall)

def _validate_adapter_files(adapter_path):
    for name in ("adapter_config.json", "adapter_model.safetensors"):
        path = Path(adapter_path) / name
        if not path.exists(): raise FileNotFoundError(f"Thiếu {path}")

_EVAL_INFER_SUBPROCESS_SRC = r'''"""ViNewsQA batch inference: Transformers + PEFT only; never imports Unsloth."""
import argparse, json, os, re, string, sys, time, unicodedata
from collections import Counter
from pathlib import Path
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
SYSTEM_PROMPT = ("Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. Chỉ trả về đúng cụm từ xuất hiện trong đoạn văn, không giải thích, không viết câu hoàn chỉnh và không thêm tiền tố như 'Đáp án:'.\n\nĐoạn văn:\n{context}")
PREFIX_RE = re.compile(r"^(đáp án|answer|câu trả lời|theo đoạn văn|trong đoạn văn)\s*[:\-]?\s*", re.IGNORECASE)
def normalize_text(text):
    text = unicodedata.normalize("NFC", text or "")
    return " ".join(text.lower().translate(str.maketrans("", "", string.punctuation)).split())
def compute_em(pred, truth): return int(normalize_text(pred) == normalize_text(truth))
def compute_f1(pred, truth):
    pt, tt = normalize_text(pred).split(), normalize_text(truth).split()
    if not pt and not tt: return 1.0
    if not pt or not tt: return 0.0
    overlap = sum((Counter(pt) & Counter(tt)).values())
    if not overlap: return 0.0
    p, r = overlap/len(pt), overlap/len(tt)
    return 2*p*r/(p+r)
def clean_prediction(raw): return PREFIX_RE.sub("", raw.strip().split("\n")[0].strip().strip("\"'" )).strip()
def messages(sample):
    return [{"role":"system", "content":SYSTEM_PROMPT.format(context=sample["context"])}, {"role":"user", "content":sample["question"]}]
def args():
    p = argparse.ArgumentParser()
    p.add_argument("--adapter-dir", required=True); p.add_argument("--samples-json", required=True); p.add_argument("--output", required=True)
    p.add_argument("--base-model", default="Qwen/Qwen2.5-1.5B-Instruct"); p.add_argument("--max-seq-length", type=int, default=1024)
    p.add_argument("--max-new-tokens", type=int, default=64); p.add_argument("--batch-size", type=int, default=16); p.add_argument("--log-every", type=int, default=50)
    return p.parse_args()
def main():
    a = args()
    if "unsloth" in sys.modules: raise RuntimeError("Eval subprocess must not import Unsloth")
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    adapter_dir = Path(a.adapter_dir)
    with open(adapter_dir/"adapter_config.json", encoding="utf-8") as f: cfg = json.load(f)
    base = cfg.get("base_model_name_or_path") or a.base_model
    if str(base).lower().startswith("unsloth/") or "qwen2.5-1.5b" in str(base).lower(): base = a.base_model
    with open(a.samples_json, encoding="utf-8") as f: samples = json.load(f)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    tokenizer = AutoTokenizer.from_pretrained(base); tokenizer.padding_side = "left"
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(base, torch_dtype=dtype).to(device)
    model = PeftModel.from_pretrained(model, str(adapter_dir), is_trainable=False).to(device); model.eval(); model.config.use_cache = True
    prompts = [tokenizer.apply_chat_template(messages(s), tokenize=False, add_generation_prompt=True) for s in samples]
    order = sorted(range(len(samples)), key=lambda i:len(prompts[i])); predictions = [None]*len(samples); start_time = time.time(); done = 0
    for start in range(0, len(order), a.batch_size):
        indices = order[start:start+a.batch_size]
        encoded = tokenizer([prompts[i] for i in indices], return_tensors="pt", padding=True, truncation=True, max_length=a.max_seq_length).to(device)
        with torch.no_grad(): output = model.generate(**encoded, max_new_tokens=a.max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        generated = output[:, encoded["input_ids"].shape[1]:]
        for row, index in enumerate(indices):
            sample = samples[index]; prediction = clean_prediction(tokenizer.decode(generated[row], skip_special_tokens=True)); golds = sample["gold_answers"]
            predictions[index] = {"id":sample["id"], "question":sample["question"], "prediction":prediction, "gold_answers":golds, "em":max(compute_em(prediction,g) for g in golds), "f1":max(compute_f1(prediction,g) for g in golds)}
        done += len(indices)
        if start == 0 or done == len(samples) or done % max(a.log_every, a.batch_size) < a.batch_size: print(f"[Sub] {done}/{len(samples)} | {done/max(time.time()-start_time,1e-3):.2f} sample/s", flush=True)
    with open(a.output, "w", encoding="utf-8") as f: json.dump(predictions, f, ensure_ascii=False)
if __name__ == "__main__": main()
'''

def _ensure_eval_infer_script():
    script = Path("eval_infer_subprocess.py")
    if not script.exists() or script.read_text(encoding="utf-8") != _EVAL_INFER_SUBPROCESS_SRC:
        script.write_text(_EVAL_INFER_SUBPROCESS_SRC, encoding="utf-8")
        print(f"Wrote clean eval subprocess: {script.resolve()}")
    return script

def _infer_predictions_subprocess(method_name, adapter_path, samples, max_seq_length):
    import shutil, sys, tempfile
    _validate_adapter_files(adapter_path); script = _ensure_eval_infer_script(); temp_dir = Path(tempfile.mkdtemp(prefix="vinewsqa_eval_"))
    samples_path, output_path = temp_dir/"samples.json", temp_dir/"predictions.json"
    with samples_path.open("w", encoding="utf-8") as f: json.dump(samples, f, ensure_ascii=False)
    cmd = [sys.executable, str(script), "--adapter-dir", str(adapter_path), "--samples-json", str(samples_path), "--output", str(output_path), "--base-model", BASE_MODEL_NAME, "--max-seq-length", str(max_seq_length), "--max-new-tokens", str(MAX_NEW_TOKENS), "--batch-size", str(INFER_BATCH_SIZE), "--log-every", str(EVAL_LOG_EVERY)]
    print(f"[Infer] {method_name}: Transformers + PEFT subprocess", flush=True)
    process = subprocess.Popen(cmd, env=dict(os.environ), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout: print(line.rstrip(), flush=True)
    finally: process.wait()
    if process.returncode:
        shutil.rmtree(temp_dir, ignore_errors=True); raise RuntimeError(f"{method_name}: eval subprocess exit {process.returncode}")
    with output_path.open(encoding="utf-8") as f: predictions = json.load(f)
    shutil.rmtree(temp_dir, ignore_errors=True); return predictions

def eval_one_adapter(method_name, adapter_path, samples, max_seq_length):
    chosen = samples[:EVAL_MAX_SAMPLES] if EVAL_MAX_SAMPLES else samples
    predictions = _infer_predictions_subprocess(method_name, adapter_path, chosen, max_seq_length)
    n = max(len(predictions), 1)
    metrics = {"method":method_name, "adapter":adapter_path, "em":round(100*sum(row["em"] for row in predictions)/n,4), "f1":round(100*sum(row["f1"] for row in predictions)/n,4), "samples":len(predictions)}
    return {"metrics":metrics, "predictions":predictions}

def export_submission_one_adapter(method_name, predictions):
    results = {row["id"]:row["prediction"] for row in predictions}
    if len(results) != len(predictions): raise RuntimeError(f"{method_name}: duplicate IDs")
    if EVAL_SPLIT == "test" and not EVAL_MAX_SAMPLES and len(results) != EXPECTED_TEST_SIZE: raise RuntimeError(f"Expected {EXPECTED_TEST_SIZE}, got {len(results)}")
    out_dir = Path(ADAPTER_SUBMISSION_DIRS[method_name]); out_dir.mkdir(parents=True, exist_ok=True); out_path = out_dir/"results.json"
    with out_path.open("w", encoding="utf-8") as f: json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"Saved submission -> {out_path}")

In [ ]:
variant_map = {variant["name"]:variant for variant in ADAPTER_VARIANTS}
all_results = {}
if RUN_METRIC_EVAL:
    if "eval_samples" not in globals(): raise NameError("Chạy cell tải dataset trước.")
    if EVAL_SPLIT == "test" and not EVAL_MAX_SAMPLES and len(eval_samples) != EXPECTED_TEST_SIZE: raise RuntimeError(f"Test eval phải có {EXPECTED_TEST_SIZE} samples")
    for method in EVAL_METHODS:
        result = eval_one_adapter(method, variant_map[method]["save_path"], eval_samples, max_seq_length)
        all_results[method] = result
        if RUN_SUBMISSION_EXPORT: export_submission_one_adapter(method, result["predictions"])
    line = "="*58
    print("\n" + line); print(f"ViNewsQA adapter comparison ({EVAL_SPLIT})"); print(line)
    print(f"{'Method':<14} | {'EM':>8} | {'F1':>8} | {'Samples':>8}"); print("-"*58)
    for method in EVAL_METHODS:
        metrics = all_results[method]["metrics"]
        print(f"{method:<14} | {metrics['em']:>7.2f}% | {metrics['f1']:>7.2f}% | {metrics['samples']:>8}")
    print(line)
    payload = {"dataset":"ViNewsQA", "eval_split":EVAL_SPLIT, "base_model":BASE_MODEL_NAME, "max_seq_length":max_seq_length, "summary":{m:r["metrics"] for m,r in all_results.items()}, "predictions":{m:r["predictions"] for m,r in all_results.items()}}
    with open(COMPARE_EVAL_PATH, "w", encoding="utf-8") as f: json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"Saved comparison -> {COMPARE_EVAL_PATH}")
else:
    print("RUN_METRIC_EVAL=False — bỏ qua đánh giá.")